# 6.2 True duplicate rate

Run this after `group_decision` is filled in `dup_review_50.csv`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

df = pd.read_csv('../data/processed/task6/dup_candidates_6_1.csv')
review = pd.read_csv('../data/processed/task6/dup_review_50.csv')

print("Dataset shape:", df.shape)
print("Review rows:", review.shape)

In [ ]:
# one decision per group (written in the first row of the group)
decisions = review.dropna(subset=['group_decision']).drop_duplicates('review_no')

print("Groups in sheet:", review['review_no'].nunique())
print("Groups reviewed:", len(decisions))
print(decisions['group_decision'].value_counts())

# stop here if the review is not finished
assert len(decisions) == review['review_no'].nunique(), "fill group_decision for every group first"
assert set(decisions['group_decision']) <= {'dup', 'not_dup', 'partial'}, "use only dup / not_dup / partial"

In [ ]:
# dup = 1, partial = 0.5, not_dup = 0
score = decisions['group_decision'].map({'dup': 1, 'partial': 0.5, 'not_dup': 0})

group_rate = score.mean()
margin = 1.96 * np.sqrt(group_rate * (1 - group_rate) / len(score))

print(f"Share of candidate groups that are real duplicates: {group_rate * 100:.1f}% (+/- {margin * 100:.1f})")

In [ ]:
# what this means for the whole dataset
n_candidate_ads = df['dup_candidate_id'].notna().sum()
n_groups = df['dup_candidate_id'].nunique()
extra_ads = n_candidate_ads - n_groups      # ads that would be removed if every group was a duplicate

dataset_rate = group_rate * extra_ads / len(df)
dataset_low = (group_rate - margin) * extra_ads / len(df)
dataset_high = (group_rate + margin) * extra_ads / len(df)

print("Candidate ads:", n_candidate_ads, "| groups:", n_groups, "| extra ads:", extra_ads)
print(f"Estimated duplicate ads in the dataset: {dataset_rate * 100:.1f}% (between {dataset_low * 100:.1f}% and {dataset_high * 100:.1f}%)")

apply_rule = dataset_rate >= 0.05
print("Apply dedup rule (rate >= 5%):", apply_rule)

In [ ]:
# does it matter if the storage was observed or imputed?
print(score.groupby(decisions['dup_key_strength']).agg(['mean', 'count']))

In [ ]:
summary = pd.DataFrame([{
    'groups_reviewed': len(decisions),
    'group_rate': round(group_rate, 3),
    'dataset_rate': round(dataset_rate, 3),
    'dataset_rate_low': round(dataset_low, 3),
    'dataset_rate_high': round(dataset_high, 3),
    'threshold': 0.05,
    'apply_rule': apply_rule,
}])
summary.to_csv('../data/processed/task6/dup_rate_6_2.csv', index=False)
summary